# Taller 1 — Fundamentos de Python y primeros pasos en redes

**ECON 64597 · Modelos de Interacciones Sociales · 2026-2**
Prof. Alvaro J. Riascos Villegas — Prof. complementaria: Gabriela Zamora

**Semana de entrega:** 33 · **Peso:** 10% de la nota final · **Grupos:** 2 a 3 personas

**Taller construido con la asistencia de IA: Claude Code (Sonnet 4.6)**  
Nota: Todo el codigo fue sugerido por la herramienta de IA, todo fue verificado por el profesor y lo mismo que la solución. Se hicieron ediciones menores.

---

## Objetivo

Al terminar este taller usted debe ser capaz de: (i) escribir funciones en Python usando
listas, diccionarios, ciclos y comprensiones; (ii) representar un grafo como lista de aristas,
lista de adyacencia y matriz de adyacencia, y moverse entre las tres representaciones;
(iii) calcular con NumPy y con `networkx` las estadisticas descriptivas basicas de una red
(grados, densidad, componentes, distancias, clustering, centralidades); (iv) medir homofilia
y contrastarla contra un modelo nulo aleatorio; y (v) interpretar economicamente los resultados.

Referencias: [J] caps. 1-3 · [EK] caps. 1-4.

## Como trabajar este cuaderno

1. Complete **todas** las celdas marcadas con `# TODO`. No borre las celdas de verificacion
   (`check(...)`): son su autodiagnostico, pero **no** son la calificacion.
2. Responda las preguntas de interpretacion en las celdas de texto indicadas. Se evaluan.
3. El cuaderno debe correr **de principio a fin** con *Kernel > Restart & Run All* sin errores
   y sin conexion a internet.
4. Entregue un `.zip` autocontenido segun las instrucciones del `README_Taller1.md`.

---

### Identificacion del grupo

| | Nombre completo | Codigo | Correo |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

### Declaracion de Uso de IA (obligatoria, seccion 9 del programa)

El uso de IA esta **permitido** en este taller. Complete honestamente:

- **Herramientas usadas:** _(p. ej. Claude, ChatGPT, Copilot, ninguna)_
- **En que puntos la usaron:** _(p. ej. E2.3 y depuracion de la Seccion 4)_
- **Que verificaron ustedes mismos:** _(p. ej. contrastamos el resultado de `nx.diameter` contra nuestro BFS)_
- **Que aprendieron / que corrigieron del output de la IA:**

> Recuerde: en la microreunion se le pedira explicar, modificar y defender cualquier linea de
> este cuaderno. "No se, lo escribio la IA" cuenta como entrega incompleta.

---
## 0. Preparacion del entorno

Esta celda importa todo lo que necesitara y define `check`, una funcion auxiliar de
verificacion. **Ejecutela primero.** Si alguna libreria falta, instalela en su entorno
(`pip install -r requirements.txt`) antes de continuar.

In [ ]:
import sys
from collections import deque

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # semilla fija: resultados reproducibles

print("Python      ", sys.version.split()[0])
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("networkx    ", nx.__version__)
print("matplotlib  ", matplotlib.__version__)


def check(nombre, condicion, ayuda=""):
    # Verificacion ligera: NO es la nota, es su autodiagnostico.
    if condicion:
        print("  OK      -", nombre)
    else:
        print("  REVISAR -", nombre, ("| " + ayuda) if ayuda else "")

---
# Seccion 1 — Fundamentos de Python con redes  *(peso: 20%)*

Un grafo no dirigido y sin pesos se puede guardar de varias formas. La mas economica es la
**lista de aristas**: una lista de parejas. Trabajaremos primero con una red juguete de 8
nodos para tener control total sobre lo que pasa.

In [ ]:
ARISTAS = [("A", "B"), ("A", "C"), ("B", "C"), ("C", "D"),
           ("D", "E"), ("E", "F"), ("F", "D"), ("G", "H")]

print("Numero de aristas:", len(ARISTAS))
ARISTAS

### E1.1 — Conjunto de nodos

Escriba `nodos(aristas)` que devuelva la **lista ordenada alfabeticamente y sin repeticiones**
de los nodos que aparecen en una lista de aristas.

*Pista:* un `set` elimina duplicados y `sorted` ordena. Se puede hacer en una sola linea con
una comprension de conjunto.

In [ ]:
def nodos(aristas):
    # TODO: devuelva la lista ordenada de nodos distintos que aparecen en `aristas`
    raise NotImplementedError


N = nodos(ARISTAS)
print(N)
check("E1.1 nodos", N == list("ABCDEFGH"), "deben ser 8 nodos ordenados de la A a la H")

### E1.2 — Lista de adyacencia

Escriba `lista_adyacencia(aristas)` que devuelva un **diccionario** `{nodo: set(vecinos)}`.
Todo nodo debe aparecer como llave, incluso si tuviera grado 0. Recuerde que el grafo es **no
dirigido**: si `(u, v)` es una arista, `u` es vecino de `v` **y** `v` es vecino de `u`.

In [ ]:
def lista_adyacencia(aristas):
    # TODO: construya {nodo: set(vecinos)}. El grafo es NO dirigido.
    raise NotImplementedError


ADJ = lista_adyacencia(ARISTAS)
for n in nodos(ARISTAS):
    print(n, "->", sorted(ADJ[n]))

check("E1.2 llaves", set(ADJ) == set(N))
check("E1.2 simetria", all(u in ADJ[v] for u in ADJ for v in ADJ[u]),
      "si v es vecino de u, u debe ser vecino de v")

### E1.3 — Grados y el lema del apreton de manos

Escriba `grados(adj)` que devuelva `{nodo: grado}` usando una **comprension de diccionario**
(una sola linea dentro de la funcion).

Luego verifique el *handshaking lemma*: la suma de los grados es igual a dos veces el numero
de aristas. Explique en una linea por que.

In [ ]:
def grados(adj):
    # TODO: comprension de diccionario {nodo: grado}
    raise NotImplementedError


G_grados = grados(ADJ)
print(G_grados)
print("Suma de grados:", sum(G_grados.values()), "| 2 * m =", 2 * len(ARISTAS))

check("E1.3 handshaking", sum(G_grados.values()) == 2 * len(ARISTAS))

**Su explicacion (1-2 lineas):**

_(escriba aqui)_

### E1.4 — Similaridad de Jaccard

Dos nodos se parecen si comparten vecinos. El **coeficiente de Jaccard** entre `u` y `v` es

$$J(u,v) = \frac{|\Gamma(u) \cap \Gamma(v)|}{|\Gamma(u) \cup \Gamma(v)|}$$

donde $\Gamma(x)$ es el conjunto de vecinos de $x$. Implementelo (devuelva `0.0` si la union
es vacia) y calculelo para los pares `(A, B)`, `(A, D)` y `(A, G)`.

In [ ]:
def jaccard(adj, u, v):
    # TODO: |vecinos(u) & vecinos(v)| / |vecinos(u) | vecinos(v)|, o 0.0 si la union es vacia
    raise NotImplementedError


for u, v in [("A", "B"), ("A", "D"), ("A", "G")]:
    print("J(%s,%s) = %.3f" % (u, v, jaccard(ADJ, u, v)))

check("E1.4 J(A,B)", abs(jaccard(ADJ, "A", "B") - 1 / 3) < 1e-9)
check("E1.4 J(A,G)", jaccard(ADJ, "A", "G") == 0.0)

### E1.5 — BFS: distancias desde un origen

Implemente una busqueda en profundidad (*breadth-first search*) que devuelva
`{nodo: distancia}` para todos los nodos **alcanzables** desde `origen`. Los nodos no
alcanzables simplemente no aparecen en el diccionario.

*Pista:* use `deque` como cola. El invariante es: cuando saca `u` de la cola, ya conoce
`dist[u]`; todo vecino no visitado esta a `dist[u] + 1`.

Esta funcion es la que despues comparara contra `networkx` — no la borre.

In [ ]:
def bfs_distancias(adj, origen):
    # TODO: BFS. Devuelva {nodo: distancia} solo para los nodos alcanzables desde `origen`.
    raise NotImplementedError


d_A = bfs_distancias(ADJ, "A")
print("Distancias desde A:", d_A)
print("Distancias desde G:", bfs_distancias(ADJ, "G"))

check("E1.5 d(A,A)=0", d_A["A"] == 0)
check("E1.5 d(A,F)=3", d_A.get("F") == 3)
check("E1.5 G no alcanzable desde A", "G" not in d_A,
      "A y G estan en componentes distintas")

---
# Seccion 2 — NumPy y la matriz de adyacencia  *(peso: 15%)*

La matriz de adyacencia $A$ de un grafo de $n$ nodos es $n \times n$ con $A_{ij} = 1$ si
$i$ y $j$ son vecinos y $0$ en otro caso. Para grafos no dirigidos $A$ es simetrica y su
diagonal es cero (no hay auto-lazos). Es la representacion que permite hacer **algebra
lineal** con la red.

### E2.1 — De lista de aristas a matriz

Escriba `matriz_adyacencia(aristas)` que devuelva la tupla `(A, orden)`, donde `A` es un
`np.ndarray` de enteros y `orden` es la lista de nodos que define las filas/columnas.

*Pista:* construya primero un diccionario `{nodo: indice}` con `enumerate`.

In [ ]:
def matriz_adyacencia(aristas):
    # TODO: devuelva (A, orden). A simetrica, de enteros, con diagonal en cero.
    raise NotImplementedError


A, orden = matriz_adyacencia(ARISTAS)
print(pd.DataFrame(A, index=orden, columns=orden))

check("E2.1 dimension", A.shape == (8, 8))
check("E2.1 simetrica", np.array_equal(A, A.T))
check("E2.1 sin auto-lazos", np.trace(A) == 0)
check("E2.1 numero de unos", A.sum() == 2 * len(ARISTAS))

### E2.2 — Grados por algebra lineal

Obtenga el vector de grados **directamente de `A`** (sin ciclos de Python) y verifique que
coincide con lo que calculo en E1.3. Guardelo en `grados_np`.

In [ ]:
# TODO: vector de grados a partir de A, sin ciclos (una sola operacion de numpy)
grados_np = ...

print(dict(zip(orden, grados_np)))
check("E2.2 coincide con E1.3",
      all(grados_np[i] == G_grados[n] for i, n in enumerate(orden)))

### E2.3 — Potencias de la matriz: contar caminos y triangulos

Un resultado clasico: la entrada $(i,j)$ de $A^k$ cuenta el numero de **recorridos** (walks)
de longitud exactamente $k$ entre $i$ y $j$.

1. Calcule `A2 = A^2` y `A3 = A^3` con `np.linalg.matrix_power`.
2. Cuente cuantos recorridos de longitud 3 hay de `A` a `D`.
3. Calcule el numero de **triangulos** del grafo. Cada triangulo se cuenta 6 veces en
   `trace(A^3)` (3 vertices de partida x 2 sentidos), de modo que el numero de triangulos es
   $\operatorname{tr}(A^3)/6$.

In [ ]:
# TODO
A2 = ...
A3 = ...

i_A, i_D = orden.index("A"), orden.index("D")
caminos_AD_3 = ...       # TODO: recorridos de longitud 3 entre A y D
n_triangulos = ...       # TODO: traza(A^3) / 6, como entero

print("Recorridos de longitud 3 entre A y D:", caminos_AD_3)
print("Numero de triangulos:", n_triangulos)

check("E2.3 triangulos", n_triangulos == 2, "ABC y DEF")
check("E2.3 diagonal de A2 = grados", np.array_equal(np.diag(A2), grados_np),
      "un recorrido de longitud 2 de i a i es ir a un vecino y volver")

**Pregunta:** la diagonal de $A^2$ resulta ser exactamente el vector de grados. Explique
por que en terminos de recorridos.

**Su respuesta:**

_(escriba aqui)_

### E2.4 — Densidad

La densidad de un grafo no dirigido simple es la fraccion de las aristas posibles que
efectivamente existen: $d = \dfrac{2m}{n(n-1)}$. Calculela **a partir de `A`**.

In [ ]:
n = ...          # TODO
m = ...          # TODO: numero de aristas, a partir de A
densidad = ...   # TODO

print("n = %d, m = %d, densidad = %.4f" % (n, m, densidad))
check("E2.4 densidad", abs(densidad - 8 / 28) < 1e-9)

---
# Seccion 3 — Pandas: cargar y explorar datos de red  *(peso: 20%)*

Pasamos a una red real y clasica: el **club de karate de Zachary** (34 miembros de un club
universitario; una arista indica interaccion social por fuera del club). El club se dividio en
dos tras un conflicto entre el instructor (`Mr. Hi`) y el administrador (`Officer`), y el
atributo `club` registra a que faccion quedo cada miembro.

La celda siguiente **genera los archivos CSV localmente** (no requiere internet). Ejecutela
tal cual.

In [ ]:
G_karate = nx.karate_club_graph()

df_aristas = pd.DataFrame(list(G_karate.edges()), columns=["source", "target"])
df_nodos = pd.DataFrame({
    "nodo": list(G_karate.nodes()),
    "club": [G_karate.nodes[i]["club"] for i in G_karate.nodes()],
})

df_aristas.to_csv("red_aristas.csv", index=False)
df_nodos.to_csv("red_nodos.csv", index=False)
print("Archivos escritos: red_aristas.csv, red_nodos.csv")

### E3.1 — Cargar los datos

Lea los dos CSV con `pd.read_csv` en `aristas` y `atributos`. Imprima la forma de cada uno y
las primeras filas. **Nunca** trabaje con datos que no haya mirado antes.

In [ ]:
aristas = ...      # TODO: leer red_aristas.csv
atributos = ...    # TODO: leer red_nodos.csv

print("aristas:", aristas.shape, "| atributos:", atributos.shape)
display(aristas.head())
display(atributos.head())

check("E3.1 aristas", aristas.shape == (78, 2))
check("E3.1 atributos", atributos.shape == (34, 2))

### E3.2 — Grados con pandas

Sin usar `networkx`, construya `df_grados`: un DataFrame con columnas `nodo` y `grado`,
ordenado de mayor a menor grado. Muestre los 5 nodos mas conectados.

*Pista:* apile las dos columnas de `aristas` con `pd.concat` y use `value_counts`.

In [ ]:
# TODO: DataFrame con columnas 'nodo' y 'grado', ordenado descendentemente por grado
df_grados = ...

display(df_grados.head())

check("E3.2 filas", len(df_grados) == 34)
check("E3.2 suma de grados", df_grados["grado"].sum() == 2 * len(aristas))
check("E3.2 nodo mas conectado", df_grados.loc[0, "nodo"] == 33)

### E3.3 — Cruce con atributos y estadisticas por grupo

1. Una `df_grados` con `atributos` (`merge` sobre `nodo`) en `df`.
2. Calcule, por faccion (`club`): numero de miembros, grado promedio, grado mediano y grado maximo.
3. Comente en una linea que sugiere la comparacion.

In [ ]:
df = ...        # TODO: merge de df_grados con atributos

# TODO: por 'club', calcular miembros, promedio, mediana y maximo del grado
resumen = ...
display(resumen)

check("E3.3 sin faltantes", df["club"].notna().all())
check("E3.3 dos facciones", len(resumen) == 2)

**Su comentario (1-2 lineas):**

_(escriba aqui)_

### E3.4 — Primera medida de homofilia

Una arista es **cruzada** si conecta miembros de facciones distintas. Calcule la fraccion de
aristas cruzadas usando solo pandas (`merge` de los atributos sobre `source` y sobre `target`).

In [ ]:
# TODO: agregue a `aristas` el club de cada extremo y calcule la fraccion de aristas cruzadas
ar = ...
ar["cruzada"] = ...
frac_cruzadas = ...

print("Aristas cruzadas: %d de %d (%.3f)" % (ar["cruzada"].sum(), len(ar), frac_cruzadas))
check("E3.4 fraccion", 0 < frac_cruzadas < 0.5)

---
# Seccion 4 — NetworkX: estadisticas descriptivas de la red  *(peso: 30%)*

Ahora si usamos la libreria estandar. Regla del curso: **primero entender, despues delegar**.
En varios puntos se le pide contrastar el resultado de `networkx` contra el codigo que usted
escribio en la Seccion 1.

### E4.1 — Construir el grafo

Construya `G` desde `aristas` con `nx.from_pandas_edgelist` y agregue el atributo `club` a
cada nodo con `nx.set_node_attributes`. Reporte $n$, $m$ y la densidad.

In [ ]:
G = ...     # TODO: nx.from_pandas_edgelist
# TODO: agregar el atributo 'club' a los nodos

print("n =", G.number_of_nodes())
print("m =", G.number_of_edges())
print("densidad = %.4f" % nx.density(G))

check("E4.1 n", G.number_of_nodes() == 34)
check("E4.1 m", G.number_of_edges() == 78)
check("E4.1 atributo club", all("club" in G.nodes[i] for i in G.nodes))

### E4.2 — Distribucion de grados

Grafique la distribucion de grados en dos paneles: (a) histograma en escala lineal;
(b) distribucion complementaria acumulada $P(K \ge k)$ en escala log-log.

Los ejes deben ir rotulados y cada panel debe tener titulo. La escala log-log es el
diagnostico visual estandar para leyes de potencia (tema de la Semana 37).

In [ ]:
grados_lista = ...   # TODO: arreglo con el grado de cada nodo

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# TODO (a): histograma de grados en el eje axes[0], con rotulos y titulo

# TODO (b): CCDF P(K >= k) en escala log-log en el eje axes[1], con rotulos y titulo

plt.tight_layout()
plt.show()

print("grado medio = %.2f | mediana = %.1f | maximo = %d"
      % (grados_lista.mean(), np.median(grados_lista), grados_lista.max()))

### E4.3 — Conectividad y componentes

Reporte si `G` es conexo, cuantas componentes conexas tiene y el tamano de la componente
gigante. Guarde en `Gc` el subgrafo inducido por la componente gigante (usela de aqui en
adelante para distancias).

In [ ]:
componentes = ...   # TODO: lista de componentes ordenada de mayor a menor
print("conexo:", ...)
print("numero de componentes:", ...)
print("tamanos:", [len(c) for c in componentes])

Gc = ...            # TODO: subgrafo de la componente gigante (use .copy())
print("componente gigante:", Gc.number_of_nodes(), "nodos")

check("E4.3 componente gigante", Gc.number_of_nodes() == 34,
      "esta red particular es conexa")

### E4.4 — Distancias: su BFS contra networkx

1. Construya la lista de adyacencia de `Gc` como diccionario `{nodo: set(vecinos)}` y calcule
   con **su** `bfs_distancias` (E1.5) las distancias desde el nodo 0.
2. Comparelas con `nx.single_source_shortest_path_length(Gc, 0)`. Deben ser identicas.
3. Reporte la distancia promedio y el diametro de `Gc`.

In [ ]:
adj_gc = ...     # TODO: {nodo: set(vecinos)} a partir de Gc

d_mio = ...      # TODO: use SU funcion bfs_distancias desde el nodo 0
d_nx = dict(nx.single_source_shortest_path_length(Gc, 0))

check("E4.4 mi BFS coincide con networkx", d_mio == d_nx)

dist_prom = ...  # TODO
diametro = ...   # TODO
print("distancia promedio = %.3f" % dist_prom)
print("diametro = %d" % diametro)

check("E4.4 diametro", diametro == 5)

### E4.5 — Clustering

Calcule el coeficiente de clustering **promedio** (`nx.average_clustering`) y el clustering
**global** o transitividad (`nx.transitivity`), e identifique los 3 nodos con mayor y los 3 con
menor clustering local (entre los que tengan grado $\ge 2$).

Recuerde: el clustering local de $i$ es la fraccion de pares de vecinos de $i$ que estan
conectados entre si. Es la formalizacion de "los amigos de mis amigos son mis amigos"
([EK] cap. 3).

In [ ]:
c_prom = ...     # TODO
c_global = ...   # TODO
print("clustering promedio = %.4f" % c_prom)
print("transitividad (global) = %.4f" % c_global)

# TODO: serie de clustering local, filtrada a nodos con grado >= 2, ordenada
cl_validos = ...

print("\nMayor clustering local:")
print(cl_validos.head(3).round(3))
print("\nMenor clustering local:")
print(cl_validos.tail(3).round(3))

### E4.6 — Centralidades

Construya `df_cent`, un DataFrame indexado por nodo con cuatro columnas: `grado`
(centralidad de grado normalizada), `cercania` (closeness), `intermediacion` (betweenness) y
`eigenvector`. Muestre el top-5 segun intermediacion.

Luego calcule la correlacion de **Spearman** entre las cuatro medidas.

In [ ]:
df_cent = pd.DataFrame({
    "grado": ...,           # TODO
    "cercania": ...,        # TODO
    "intermediacion": ...,  # TODO
    "eigenvector": ...,     # TODO  (use max_iter=1000)
})

display(df_cent.sort_values("intermediacion", ascending=False).head(5).round(4))
print("\nCorrelacion de Spearman entre centralidades:")
display(df_cent.corr(method="spearman").round(3))

check("E4.6 columnas", list(df_cent.columns) == ["grado", "cercania",
                                                "intermediacion", "eigenvector"])

**Interpretacion (2-4 lineas):** compare el ranking por grado con el ranking por
intermediacion. Identifique al menos un nodo que suba mucho de posicion y explique por que.

_(escriba aqui)_

### E4.7 — Visualizacion

Dibuje la red con `nx.spring_layout(G, seed=42)`, coloreando los nodos por faccion y con
tamano proporcional a la centralidad de intermediacion. Incluya titulo y leyenda de colores.

In [ ]:
pos = nx.spring_layout(G, seed=42)

# TODO: vector de colores segun el atributo 'club'
colores = ...
# TODO: vector de tamanos proporcional a la intermediacion (escalelo para que se vea)
tam = ...

fig, ax = plt.subplots(figsize=(8, 7))
# TODO: dibuje aristas, nodos y etiquetas; agregue titulo y leyenda
plt.tight_layout()
plt.show()

---
# Seccion 5 — Homofilia contra un modelo nulo  *(peso: 15%)*

En E3.4 usted encontro que pocas aristas cruzan las facciones. Pero **un numero bajo no prueba
nada por si solo**: si un grupo es pequeno, habra pocas aristas cruzadas incluso con
emparejamiento puramente aleatorio. Necesitamos un **modelo nulo**.

Siguiendo [EK] cap. 4: si cada nodo escogiera vecinos al azar, con fracciones poblacionales
$p$ y $q = 1-p$, la probabilidad de que una arista sea cruzada seria $2pq$. La red exhibe
**homofilia** si la fraccion observada de aristas cruzadas es sustancialmente **menor** que $2pq$.

### E5.1 — Test analitico de homofilia

Calcule $p$, $q$, el valor esperado $2pq$ y comparelo con `frac_cruzadas` de E3.4.

In [ ]:
p = ...          # TODO: fraccion de la faccion 'Mr. Hi'
q = ...
esperado = ...   # TODO: 2pq

print("p (Mr. Hi)      = %.4f" % p)
print("q (Officer)     = %.4f" % q)
print("2pq esperado    = %.4f" % esperado)
print("observado       = %.4f" % frac_cruzadas)
print("razon obs/esp   = %.3f" % (frac_cruzadas / esperado))

check("E5.1 hay homofilia", frac_cruzadas < esperado)

### E5.2 — Test por permutaciones (modelo nulo empirico)

El calculo anterior ignora la estructura de grados. Un modelo nulo mas honesto: **mantenga
fija la red** y permute aleatoriamente las etiquetas `club` entre los nodos 5.000 veces,
recalculando cada vez la fraccion de aristas cruzadas. Eso genera la distribucion de la
estadistica bajo la hipotesis nula de "las etiquetas no tienen relacion con la estructura".

1. Escriba la simulacion (use `rng.permutation`).
2. Reporte el p-valor empirico de una cola: la proporcion de permutaciones con fraccion
   cruzada **menor o igual** a la observada.
3. Grafique la distribucion nula con una linea vertical en el valor observado.

In [ ]:
etiquetas = atributos.set_index("nodo")["club"]
src = aristas["source"].values
tgt = aristas["target"].values
lab = etiquetas.loc[sorted(etiquetas.index)].values

n_sim = 5000
simuladas = np.empty(n_sim)
for s in range(n_sim):
    # TODO: permute las etiquetas y guarde la fraccion de aristas cruzadas resultante
    simuladas[s] = ...

p_valor = ...   # TODO: proporcion de simulaciones <= valor observado
print("observado = %.4f" % frac_cruzadas)
print("media bajo el nulo = %.4f" % simuladas.mean())
print("p-valor empirico (una cola) = %.4f" % p_valor)

fig, ax = plt.subplots(figsize=(7, 4))
# TODO: histograma de `simuladas` con una linea vertical en el valor observado
plt.tight_layout()
plt.show()

check("E5.2 significativo al 5%", p_valor < 0.05)

### E5.3 — Comparacion con un grafo de Erdos-Renyi

Genere 200 grafos $G(n, p)$ con el mismo $n$ y la misma densidad que `G`, y compare el
clustering promedio y la distancia promedio observados contra la distribucion aleatoria.
Descarte las replicas que resulten no conexas al calcular la distancia promedio.

In [ ]:
n_g = G.number_of_nodes()
p_g = nx.density(G)

cl_sim, d_sim = [], []
for s in range(200):
    H = ...   # TODO: grafo aleatorio G(n, p) con la misma densidad
    # TODO: acumule clustering promedio; y distancia promedio solo si H es conexo
    pass

print("clustering:  observado = %.4f | aleatorio = %.4f" % (c_prom, np.mean(cl_sim)))
print("distancia:   observado = %.4f | aleatorio = %.4f" % (dist_prom, np.mean(d_sim)))
print("(%d de 200 replicas aleatorias resultaron conexas)" % len(d_sim))

**Su lectura (2-4 lineas):** compare ambas magnitudes y diga que propiedad de las redes
sociales reales queda en evidencia.

_(escriba aqui)_

---
# Seccion 6 — Preguntas de interpretacion  *(peso: 15%)*

Responda en las celdas de texto. Se evalua el **razonamiento economico y estadistico**, no la
extension: dos o tres parrafos bien argumentados valen mas que una pagina. Prepare estas
respuestas para defenderlas oralmente en la microreunion.

**P1.** Suponga que un investigador solo observa la lista de aristas y **no** el atributo
`club`. Con las herramientas de este taller, describa como intentaria detectar que la red
esta dividida en dos grupos. Que evidencia buscaria y cual seria la principal limitacion de
su aproximacion?

**Su respuesta:**

_(escriba aqui)_

**P2.** El test de E5.2 permuta las etiquetas manteniendo fija la red. Que hipotesis
nula esta contrastando exactamente? Por que ese modelo nulo es preferible al calculo
analitico $2pq$ de E5.1?

**Su respuesta:**

_(escriba aqui)_

**P3.** Los datos del club de karate provienen de la observacion directa de un
antropologo durante dos anos. Compare esa forma de recoleccion con la de una red extraida hoy
de una plataforma digital (p. ej. seguidores en una red social). Mencione al menos dos sesgos
de medicion y una implicacion para la validez de los resultados.

**Su respuesta:**

_(escriba aqui)_

**P4.** Esta red identifica a 34 personas reales con sus relaciones sociales y su
faccion politica interna. Suponga que quiere publicar una version de estos datos. Que riesgo
de re-identificacion existe si solo borra los nombres y deja la estructura? Relacione su
respuesta con el tema de privacidad de la Semana 38.

**Su respuesta:**

_(escriba aqui)_

---
# Bonus (opcional, hasta +0.5 sobre 5.0)

Escoja **uno**:

- **B1.** Repita E5.3 usando el **modelo de configuracion** (`nx.configuration_model`) en lugar
  de Erdos-Renyi, para preservar la secuencia de grados observada. Cambia la conclusion sobre
  el clustering? Por que?
- **B2.** Implemente desde cero (sin `networkx`) el algoritmo de particion espectral: calcule
  el laplaciano $L = D - A$, obtenga el vector de Fiedler (segundo vector propio mas pequeno)
  y particione segun su signo. Compare la particion resultante con el atributo `club` real y
  reporte cuantos nodos clasifica correctamente.
- **B3.** Elimine iterativamente el nodo de mayor grado y el nodo de mayor intermediacion, y
  grafique como cae el tamano de la componente gigante en cada caso. Cual estrategia de ataque
  fragmenta la red mas rapido y que implica eso para la robustez de la red?

In [ ]:
# Espacio para el bonus (opcional)

---
## Cierre — lista de chequeo antes de entregar

- [ ] Tabla de integrantes diligenciada.
- [ ] Declaracion de Uso de IA diligenciada.
- [ ] *Kernel > Restart & Run All* corre completo y sin errores.
- [ ] Todas las verificaciones `check(...)` imprimen OK.
- [ ] Las preguntas de la Seccion 6 estan respondidas.
- [ ] El `.zip` incluye el `.ipynb` ejecutado, `requirements.txt` y los CSV generados.
- [ ] Todos los integrantes pueden explicar **cualquier** celda del cuaderno.